## Speed Up Your Python Program With Concurrency

Concurrency refers to the ability of a program to manage multiple tasks at once, improving performance and responsiveness. It encompasses different models: 
- threading (Thread)
- asynchronous tasks (Task)
- multiprocessing (Process)

Each offering unique benefits and trade-offs. In Python, threads and asynchronous tasks facilitate concurrency on a single processor, while ***multiprocessing allows for true parallelism*** by utilizing multiple CPU cores.

See:
- https://realpython.com/python-concurrency/
- https://realpython.com/python-concurrency/#speeding-up-a-cpu-bound-program
- https://realpython.com/intro-to-python-threading/

## Speeding Up an CPU-Bound Program

CPU-bound problem performs fewer I/O operations, and its total execution time depends on how quickly it can process the required data.


### Synchronous Version
This version of your program doesn’t use concurrency at all.

In [15]:
# Calculate Fibonacci numbers
def fib(n):
    return n if n < 2 else fib(n - 2) + fib(n - 1)
    
for n in range(1, 11):
    #print(f"fib({n}) = {fib(n)}")   # simple print format
    print(f"fib({n:>2}) = {fib(n)}") # better print format

fib( 1) = 1
fib( 2) = 1
fib( 3) = 2
fib( 4) = 3
fib( 5) = 5
fib( 6) = 8
fib( 7) = 13
fib( 8) = 21
fib( 9) = 34
fib(10) = 55


In [16]:
# Calculate Fibonacci numbers for performance mesurement purpose
import time

def fib(n):
    return n if n < 2 else fib(n - 2) + fib(n - 1)

start_time = time.perf_counter()
for _ in range(20):
    fib(35)
duration = time.perf_counter() - start_time
print(f"Computed in {duration} seconds")

Computed in 33.84663134899995 seconds


### Multi-Threaded Version
Using concurrent.futures and threading modules. A Thread-Pool-Executor, you’ll end up with these three components:
- Thread
- Pool
- Executor

With a CPU-bound problem, there’s no waiting. The CPU is cranking away as fast as it can to finish the problem. In Python, both threads and asynchronous tasks run on the same CPU in the same process. This means that the one CPU is doing all of the work of the non-concurrent code plus the extra work of setting up threads or tasks. As the result, simple multi-threaded version might actually slow the program down.

In [17]:
import time
from concurrent.futures import ThreadPoolExecutor

def fib(n):
    return n if n < 2 else fib(n - 2) + fib(n - 1)

start_time = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor: # a pool of 5 threads
    executor.map(fib, [35] * 20) # [35] is the parameter list passed to the function.
duration = time.perf_counter() - start_time
print(f"Computed in {duration} seconds")    

Computed in 32.71402534799995 seconds


### Asynchronous Version

Using asyncio (Async IO) library.

See https://realpython.com/python-concurrency/#asynchronous-version_1

As expected, the asynchronous approach is the slowest for a CPU-bound problem because there are no I/O operations involved here, there’s nothing to wait for. The overhead of the event loop and context switching at every single await statement slows down the total execution substantially.

In [19]:
import asyncio
import time

async def main():
    start_time = time.perf_counter()
    tasks = [fib(35) for _ in range(20)]
    await asyncio.gather(*tasks, return_exceptions=True)
    duration = time.perf_counter() - start_time
    print(f"Computed in {duration} seconds")

async def fib(n):
    return n if n < 2 else await fib(n - 2) + await fib(n - 1)

await main()

Computed in 76.03359249000005 seconds


### Process-Based Version

Using ProcessPoolExecutor

See https://realpython.com/python-concurrency/#process-based-version

The multiprocessing module, along with the corresponding wrappers in concurrent.futures, was designed to break down single CPU barrier and run your code across multiple CPUs. At a high level, it does this by creating a new instance of the Python interpreter to run on each CPU and then farming out part of your program to run on it.

Unlike the other concurrency models, process-based parallelism is explicitly designed to share heavy CPU workloads across multiple CPUs.

There are some drawbacks to using multiprocessing that don’t really show up in a simple example like this one. For example, dividing your problem into segments so each processor can operate independently can sometimes be difficult.

In [20]:
import time
from concurrent.futures import ProcessPoolExecutor

def main():
    start_time = time.perf_counter()
    with ProcessPoolExecutor() as executor:
        executor.map(fib, [35] * 20)
    duration = time.perf_counter() - start_time
    print(f"Computed in {duration} seconds")

def fib(n):
    return n if n < 2 else fib(n - 2) + fib(n - 1)

main()

Computed in 5.1125904380000975 seconds
